# Stateful operators

Stateful operators bring the real fun to Kafi Streams. The most interesting operators are of course `join_equi()` and `join()` (equi join and general/non-equi join) and `group_by_agg()` (group by + aggregate). `agg()` is just a special case of `group_by_agg()` (obviously, without grouping).

Kafi Streams also already supports the stateful set operators `distinct()`, `union()`, `intersect()` and `minus()`.

## Overview

  * [join_equi()](#join_equi-operator)
  * [join()](#join-operator)
  * [group_by_agg()](#group_by_agg-operator)
    * [group_by_sum()](#group_by_sum-operator)
    * [group_by_max()](#group_by_max-operator)
    * [group_by_min()](#group_by_min-operator)
    * [group_by_avg()](#group_by_avg-operator)
    * [group_by_count()](#group_by_count-operator)
  * [agg()](#agg-operator)
    * [sum()](#sum-operator)
    * [max()](#max-operator)
    * [min()](#min-operator)
    * [avg()](#avg-operator)
    * [count()](#count-operator)
  * [distinct()](#distinct-operator)
  * [union()](#union-operator)
  * [intersect()](#intersect-operator)
  * [minus()](#minus-operator)


## Preparation

Before we start off, we first prepare for the examples to follow:

In [2]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


<a id="join_equi-operator"></a>
## join_equi()

Probably the most popular stateful operator - `join_equi()` is the equi join operator of Kafi Streams:
```python
join_equi(right_tn, left_select_fun, right_select_fun, project_fun, **kwargs):
```

To its parameters:
* `right_tn`: the right side of the join.
* `left_select_fun: l_r -> any`: the selection function getting an input record an returning the left join key
* `right_select_fun: r_r -> any`: the selection function for the right join key
* `project_fun: l_r, r_r -> r`: the projection function; getting both the left and right input records and returning the output (=projection) record of the join.

Here is an example:

In [15]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
)

built_tn = Tn.build(
    click_tn
    .join_equi(customer_tn,
               left_select_fun=lambda l_r: l_r["customer_id"],
               right_select_fun=lambda r_r: r_r["id"],
               project_fun=lambda l_r, r_r: {"customer_id": l_r["customer_id"],
                                             "view_time": l_r["view_time"],
                                             "name": r_r["name"]})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Input (clicks):
{'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618958910}}
{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618968910}}
{'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}

Input (customers):
{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}}
{'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}

Output:
{'customer_id': 42, 'view_time': 23, 'name': 'Betty Graham MD'}
{'customer_id': 42, 'view_time': 67, 'name': 'Betty Graham MD'}


In this example topology, we join the clicks and customers as before in the [Quickstart](../quickstart.ipynb).

The example data consists of three clicks and two customers. Two of the clicks match a customer from the right side (`customer_id` = `42`) and thus we receive both clicks (joined with the `name`) in the output.

<a id="join-operator"></a>
## join()

While `join_equi()` might be the most popular stateful operator, Kafi Streams also offers a general/non-equi join with the `join()` operator. Whereas with `join_equi()`, you can only select a key from each side of the join, `join()` allows you to specify any predicate:
```python
join(right_tn, predicate_fun, project_fun, **kwargs)
```

To the parameters:
* `right_tn`: the right side of the join
* `predicate_fun: l_r, r_r -> bool`: the join predicate - a function getting both the left and right input records and returning a bool.
* `project_fun: l_r, r_r -> r`: the projection function; getting both the left and right input records and returning the output (=projection) of the join.

As in all databases as well, non-equi join operator (`join()`) is, on the one hand, more flexible than `join_equi()`, but on the other hand much less performant:
* `join_equi` is based on hash maps (complexity `O(1)` per row)
* `join` cannot use hash maps (`O(N)`)

So whenever you can, use `join_equi()`, and `join()` only if you need the extra flexibility or the predicate can be calculated quickly enough (e.g. in case one of the two sides of the join always stays very small).

An example is in order for the `join()` operator as well:

In [16]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
)

built_tn = Tn.build(
    click_tn
    .join(customer_tn,
          predicate_fun=lambda l_r, r_r: l_r["customer_id"] == r_r["id"] and l_r["view_time"] > 60,
          project_fun=lambda l_r, r_r: {"customer_id": l_r["customer_id"],
                                        "view_time": l_r["view_time"],
                                        "name": r_r["name"]})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Input (clicks):
{'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618958910}}
{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618968910}}
{'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}

Input (customers):
{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}}
{'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}

Output:
{'customer_id': 42, 'view_time': 67, 'name': 'Betty Graham MD'}


Except for the change from `join_equi()` to `join`, the topology is the same as in the previous example for `join_equi()`. The join predicate also works a little differently to the two selections in the `join_equi()` example: This time, we do not only match the customer IDs but also add the condition that the `view_time` of the click must be greater than `60`.

The example data is the same as in the previous example for `join_equi()`. But the output is different, because even though the second click of customer `42` does match the customer ID of one of the customers, its `view_time` is not greater than `60`. Thus we only receive one output record.